# HW 4 — Regex with AI: Generate It, Then Verify It

## Part A — Generate It



### A1. Student and Filing Information

In [6]:
# --- Cell 1: you and your claimed filing ---

name        = "리오디노 라이한"
student_id  = "50261893"
company     = "Federal Signal Corporation"
fiscal_year = "FY2024"
claimed_url = "https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm"

print("Name:", name)
print("Student ID:", student_id)
print("Company:", company)
print("Fiscal Year:", fiscal_year)
print("Exhibit 21 URL:", claimed_url)

Name: 리오디노 라이한
Student ID: 50261893
Company: Federal Signal Corporation
Fiscal Year: FY2024
Exhibit 21 URL: https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm


### A2. Download SEC Exhibit 21

In [8]:
# --- Cell 2: fetch the exhibit ---

import requests

URL = "https://www.sec.gov/Archives/edgar/data/277509/000027750925000012/fss-20241231x10kexhx21.htm"

headers = {
    "User-Agent": "Riodino Raihan riodinoraihan@gmail.com"
}

response = requests.get(URL, headers=headers)

print("Status code:", response.status_code)
print("Downloaded characters:", len(response.text))

html = response.text

Status code: 200
Downloaded characters: 21884


### A3. Inspect the Downloaded HTML

In [9]:
print(html[:500])

<DOCUMENT>
<TYPE>EX-21
<SEQUENCE>3
<FILENAME>fss-20241231x10kexhx21.htm
<DESCRIPTION>SUBSIDIARIES OF THE REGISTRANT
<TEXT>
<html><head>
<!-- Document created using Wdesk -->
<!-- Copyright 2025 Workiva -->
<title>Document</title></head><body><div id="i0b29d3f9341a44a1b83bbcc9e50cc3e3_1"></div><div style="min-height:42.75pt;width:100%"><div><font><br></font></div></div><div style="text-align:right"><font style="color:#000000;font-family:'Times New Roman',sans-serif;font-size:10pt;font-weight:700;


### A4. AI Tool, Prompt, and Generated Code

#### AI Tool, Exact Prompt, and Exact Code
**AI Tool / Model:** [nama model Gemini yang Anda gunakan]

**Exact Prompt:**

[PASTE prompt yang benar-benar Anda kirim ke Gemini]

**Exact AI-Generated Code:**

[PASTE kode persis yang diberikan Gemini]

#### Generated Regex Code

In [10]:
import re
import html as html_lib

# (Assuming the `html` variable is already defined and contains the raw HTML string)
# html = "..."

def extract_subsidiaries(html_content):
    # Regex patterns for finding rows, cells, and removing HTML tags
    # We account for possible whitespace in closing tags (e.g., </tr >)
    row_regex = re.compile(r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>', re.IGNORECASE | re.DOTALL)
    cell_regex = re.compile(r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>', re.IGNORECASE | re.DOTALL)
    tag_regex = re.compile(r'<[^>]+>')
    space_regex = re.compile(r'\s+')
    
    extracted_pairs = []
    
    # 1. Extract all <tr> blocks
    rows = row_regex.findall(html_content)
    
    for row in rows:
        # 2. Extract all <td> blocks within the current row
        cells = cell_regex.findall(row)
        
        cleaned_cells = []
        for cell in cells:
            # Replace HTML tags with a space to prevent words from fusing together
            text = tag_regex.sub(' ', cell)
            
            # Unescape HTML entities (converts &amp; to &, &#160; to space, etc.)
            text = html_lib.unescape(text)
            
            # Replace multiple whitespace characters (newlines, tabs, etc.) with a single space
            text = space_regex.sub(' ', text).strip()
            
            # If the cell contains text after cleaning, add it to our valid cells list
            if text:
                cleaned_cells.append(text)
        
        # 3. Process rows with at least 2 non-empty cells
        if len(cleaned_cells) >= 2:
            # First relevant cell is the Subsidiary Name, the last is the Location
            subsidiary_name = cleaned_cells[0]
            location = cleaned_cells[-1]
            
            # Ignore table header rows that usually contain words like "Subsidiary", "Name", "Jurisdiction"
            name_lower = subsidiary_name.lower()
            loc_lower = location.lower()
            if ("subsidiary" in name_lower or "name of" in name_lower or 
                "jurisdiction" in loc_lower or "state" in loc_lower):
                continue
                
            extracted_pairs.append((subsidiary_name, location))
            
    return extracted_pairs

# --- Execution ---
pairs = extract_subsidiaries(html)

# --- Output Results ---
print(f"Total extracted pairs: {len(pairs)}\n")

print("First 10 pairs:")
for i, (name, loc) in enumerate(pairs[:10], start=1):
    print(f"{i}. {name} | {loc}")

Total extracted pairs: 33

First 10 pairs:
1. Crysteel Manufacturing, Inc. | Minnesota
2. Deist Industries, LLC | Delaware
3. Elgin Sweeper Company | Delaware
4. Federal Signal of Texas Corp. | Texas
5. Federal Signal UK Holdings Limited | United Kingdom
6. Federal Signal VAMA, S.A. | Spain
7. FS Depot, LLC | Wisconsin
8. FST Canada Inc. | Canada
9. FST of Tennessee, Inc. | Tennessee
10. GenNx/TBEI Intermediate Co. | Delaware


## Part B — Verify It



### B1. Ground Truth

I manually counted the named subsidiary rows in the Exhibit 21 table. The manual count is **33 subsidiaries**, excluding the table header and the footnote below the table.

**Manual count = 33**  
**AI-generated regex count = 33**

Therefore, the counts match.

### B2. Count Verification

In [ ]:
print("Manual count:", 33)
print("AI-generated regex count:", len(pairs))
print("Counts match:", len(pairs) == 33)

### B3. Complete Extraction Result

In [11]:
print("All extracted pairs:")
for i, pair in enumerate(pairs, start=1):
    print(f"{i}. {pair[0]} | {pair[1]}")

All extracted pairs:
1. Crysteel Manufacturing, Inc. | Minnesota
2. Deist Industries, LLC | Delaware
3. Elgin Sweeper Company | Delaware
4. Federal Signal of Texas Corp. | Texas
5. Federal Signal UK Holdings Limited | United Kingdom
6. Federal Signal VAMA, S.A. | Spain
7. FS Depot, LLC | Wisconsin
8. FST Canada Inc. | Canada
9. FST of Tennessee, Inc. | Tennessee
10. GenNx/TBEI Intermediate Co. | Delaware
11. Ground Force Manufacturing LLC | Delaware
12. Guzzler Manufacturing, Inc. | Alabama
13. HighMark Traffic Services, Inc. | Montana
14. Jetstream of Houston, Inc. | Delaware
15. Jetstream of Houston LLP | Texas
16. Joe Johnson Equipment LLC | Delaware
17. Mark Rite Lines Equipment Company, Inc. | Delaware
18. Northend Truck Equipment, LLC | Washington
19. OSW Equipment & Repair, LLC | Washington
20. Ox Bodies, Inc. | Alabama
21. Rugby Manufacturing Company | Oregon
22. Tishomingo Acquisition, LLC | Delaware
23. Travis Acquisition LLC | Delaware
24. Travis Body and Trailer, Inc. | Tex

### B4. First and Last Row Verification

In [12]:
print("First extracted row:")
print(pairs[0])

print("\nLast extracted row:")
print(pairs[-1])

First extracted row:
('Crysteel Manufacturing, Inc.', 'Minnesota')

Last extracted row:
('Work Equipment Ltd.', 'Canada')


### B5. Awkward Row Tests

#### Test 1 — Ampersand in Subsidiary Name

In [13]:
for pair in pairs:
    if "OSW Equipment" in pair[0]:
        print(pair)

('OSW Equipment & Repair, LLC', 'Washington')


#### Test 2 — Ampersand in Another Subsidiary Name

In [14]:
for pair in pairs:
    if "Truck Bodies & Equipment" in pair[0]:
        print(pair)

('Truck Bodies & Equipment International, Inc.', 'Delaware')


#### Test 3 — Parentheses in Subsidiary Name

In [15]:
for pair in pairs:
    if "Victor Industrial Equipment" in pair[0]:
        print(pair)

('Victor Industrial Equipment (PTY) Limited', 'South Africa')


### B6. Missing-Location Edge Case

In [16]:
test_html = """
<table>
<tr>
<td>Test Subsidiary</td>
<td></td>
</tr>
</table>
"""

test_pairs = extract_subsidiaries(test_html)

print("Missing-location test:")
print(test_pairs)

Missing-location test:
[]


### B7. Path 2 — Proving It

I established the ground truth manually by counting the subsidiary rows in Federal Signal Corporation's FY2024 Exhibit 21. The filing contains 33 named subsidiary rows, and the AI-generated regex also extracted 33 pairs, so the counts match.

The first extracted pair was `Crysteel Manufacturing, Inc. — Minnesota`, which matches the first subsidiary row in the filing. The last extracted pair was `Work Equipment Ltd. — Canada`, which also matches the last subsidiary row.

I deliberately tested several awkward cases. `OSW Equipment & Repair, LLC — Washington` and `Truck Bodies & Equipment International, Inc. — Delaware` confirmed that the regex handled subsidiary names containing an ampersand (`&`). `Victor Industrial Equipment (PTY) Limited — South Africa` confirmed that parentheses in a subsidiary name were also handled correctly.

The original filing does not contain a named subsidiary row with a missing location, so I did not invent one. Instead, I created a small deliberate edge-case test containing a subsidiary with an empty location cell. The result was an empty list (`[]`), showing that the code does not return an incomplete subsidiary-location pair as valid data.

The regex could still break on another company's Exhibit 21 if the filing used a substantially different HTML structure, such as `<th>` elements instead of `<td>` elements, nested tables, merged cells using `rowspan` or `colspan`, or a different arrangement of the subsidiary and jurisdiction columns.

## Part C — Fixing and Explaining It



### C1. Fix

No fix was needed. The original AI-generated regex successfully extracted all 33 subsidiary-location pairs from the Federal Signal Corporation FY2024 Exhibit 21. The first and last rows were correctly extracted, and the deliberately tested awkward cases were also handled correctly.

Because the verification results matched the manually established ground truth, I did not modify the AI-generated regex.

### C2. Regex Explanation



#### C2.1 Row Pattern

In [23]:
row_regex = re.compile(
    r'<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>',
    re.IGNORECASE | re.DOTALL
)

The row pattern is:

`<\s*tr[^>]*>(.*?)<\s*/\s*tr\s*>`

- `<\s*tr` identifies an opening `<tr>` tag.
- `[^>]*` allows attributes inside the opening tag.
- `(.*?)` captures the row contents.
- `<\s*/\s*tr\s*>` matches the closing `</tr>` tag.
- `re.IGNORECASE` allows different capitalization of HTML tags.
- `re.DOTALL` allows the match to include line breaks.

#### C2.2 Cell Pattern

In [24]:
cell_regex = re.compile(
    r'<\s*td[^>]*>(.*?)<\s*/\s*td\s*>',
    re.IGNORECASE | re.DOTALL
)

The cell pattern is:

`<\s*td[^>]*>(.*?)<\s*/\s*td\s*>`

- `td` identifies an HTML table cell.
- `[^>]*` allows attributes inside the `<td>` opening tag.
- `(.*?)` captures the contents of the cell without taking more text than necessary.
- The closing part matches `</td>`.
- `re.IGNORECASE` allows different capitalization of HTML tags.
- `re.DOTALL` allows the cell contents to include line breaks.

#### C2.3 HTML Tag Removal

In [25]:
tag_regex = re.compile(r'<[^>]+>')

The pattern:

`<[^>]+>`

is used to remove HTML tags from the cell content.

- `<` marks the beginning of an HTML tag.
- `[^>]+` means one or more characters that are not `>`.
- `>` marks the end of the tag.

Therefore, HTML tags such as `<font>`, `<div>`, or other nested tags can be removed while keeping the text inside them.

#### C2.4 Whitespace Normalization

In [26]:
space_regex = re.compile(r'\s+')

The pattern:

`\s+`

matches one or more whitespace characters.

Here, `\s` represents whitespace such as spaces, tabs, and line breaks, while `+` means one or more occurrences. Replacing these matches with a single space makes the extracted subsidiary names and locations cleaner and more consistent.

#### C2.5 Selecting the Name and Location

After extracting the `<td>` cells, the code cleans every cell and keeps only non-empty cells. The first non-empty cell is assigned as the subsidiary name, while the last non-empty cell is assigned as the location.

This is important for this filing because the table contains empty cells used for spacing. Therefore, the code does not simply assume that the second physical `<td>` is always the location.

### C3. Reflection

The AI-generated regex worked correctly on the Federal Signal Corporation FY2024 Exhibit 21, so I did not find a failure that needed to be fixed. The verification process was important because I could confirm the result by comparing the extracted count with my manual ground truth and by checking the first, last, and awkward rows. Even if I could not understand the regex syntax, I could still detect a problem by comparing the AI output with the original filing and checking whether any subsidiaries were missing or incorrectly extracted. I would also test the same approach on a differently structured Exhibit 21 because a regex that works for one HTML structure may not work for every SEC filing.